# PatchCore CPU walkthrough
This notebook calls the reusable PatchCore workflow. It does not contain a second implementation.

**DEBUG METRICS — NOT FOR REPORTING**

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import torch

repository_root = Path.cwd().resolve()
if repository_root.name == 'notebooks':
    repository_root = repository_root.parent
os.chdir(str(repository_root))
sys.path.insert(0, str(repository_root))
from scripts.smoke_test_patchcore import PatchCoreSmokeTester

config_path = repository_root / 'configs/patchcore_cpu_smoke.yaml'
print({'python': platform.python_version(), 'torch': torch.__version__, 'device': 'cpu'})

/home/hanhpm/miniconda3/envs/robustvisionad_patchcore/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.8.20', 'torch': '2.4.1+cu121', 'device': 'cpu'}


## Dataset, features, embeddings, memory bank, coreset, NN scoring, and anomaly map
The shared smoke workflow validates each stage and returns its observable shapes and scores.

In [2]:
report = PatchCoreSmokeTester().run(config_path, fit_memory_bank=True)
assert report['device'] == 'cpu'
assert report['debug_metrics_not_for_reporting'] is True
assert report['nan'] is False and report['inf'] is False
print(json.dumps(report, indent=2, sort_keys=True))

/home/hanhpm/miniconda3/envs/robustvisionad_patchcore/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hanhpm/miniconda3/envs/robustvisionad_patchcore/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Computing support features...:   0%|          | 0/10 [00:00<?, ?it/s]

Computing support features...:  10%|█         | 1/10 [00:00<00:00,  9.67it/s]

Computing support features...:  30%|███       | 3/10 [00:00<00:00, 10.15it/s]

Computing support features...:  50%|█████     | 5/10 [00:00<00:00,  9.85it/s]

...

Inferring...:  70%|███████   | 7/10 [00:00<00:00,  9.03it/s]

Inferring...:  80%|████████  | 8/10 [00:00<00:00,  8.89it/s]

Inferring...:  90%|█████████ | 9/10 [00:00<00:00,  8.70it/s]

Inferring...: 100%|██████████| 10/10 [00:01<00:00,  8.54it/s]

{
  "anomaly_anomaly_map_shape": [
    128,
    128
  ],
  "anomaly_image_score": 0.5247759819030762,
  "anomaly_nn_distance_shape": [
    256,
    1
  ],
  "anomaly_patch_score_max": 0.5247759819030762,
  "anomaly_patch_score_min": 0.0007729530334472656,
  "backbone_eval": true,
  "backbone_frozen": true,
  "coreset_ratio": 0.01,
  "debug_au_pro": 0.8614374706166832,
  "debug_au_pro_fpr_limit": 0.3,
  "debug_image_auroc": 1.0,
  "debug_metrics_not_for_reporting": true,
  "debug_pixel_auroc": 0.9510548995666066,
  "device": "cpu",
  "embedding_dimension": 256,
  "evaluation_anomaly_images": 5,
  "evaluation_normal_images": 5,
  "inf": false,
  "input_shape": [
    1,
    3,
    128,
    128
  ],
  "layer2_shape": [
    1,
    512,
    16,
    16
  ],
  "layer3_shape": [
    1,
    1024,
    8,
    8
  ],
  "memory_bank_shape": [
    25,
    256
  ],
  "nan": false,
  "normal_anomaly_map_shape": [
    128,
    128
  ],
  "normal_image_score": 0.40839219093322754,
  "normal_nn_distance_s

## Interpretation
The anomaly-map shape must match the configured input size. The reported I-AUROC, P-AUROC, and AU-PRO use only a tiny balanced subset and are **DEBUG METRICS — NOT FOR REPORTING**. AU-PRO uses the documented FPR limit of 0.30.